
# Dense Information Retrieval

### Dataset: CORD-19 (COVID-19 Open Research Dataset)


### Traditional sparse retrieval

```text
Document → vocabulary-sized sparse vector → inverted index → lexical matching
```

### Dense retrieval

```text
Document → embedding model → dense vector → vector index → semantic similarity
```

### RAG

```text
Query
  ↓
Dense / Hybrid Retriever
  ↓
Top-k relevant passages
  ↓
LLM
  ↓
Generated answer
```


In [2]:

# Install packages
!pip install pandas==2.2.3
!pip install "numpy<2.3"
!pip -q install -U sentence-transformers faiss-cpu rank-bm25 scikit-learn matplotlib seaborn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.9 MB/s eta 0:00:00


In [3]:

# Imports and configuration

import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Number of CORD-19 papers used in the demo.
# 5,000–10,000 is a good classroom setting for Colab.
MAX_DOCS = 10_000

print("FAISS version:", faiss.__version__)


FAISS version: 1.15.1


In [4]:
# Load CORD-19 metadata

!wget -q https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2022-06-02/metadata.csv \
    -O /content/cord19_metadata.csv

CORD19_PATH = "/content/cord19_metadata.csv"

if not os.path.exists(CORD19_PATH):
    print("metadata.csv was not found.")
    print("Upload it to Colab, then rerun this cell.")
else:
    print("Found:", CORD19_PATH)


Found: /content/cord19_metadata.csv


In [5]:
# Read and clean CORD-19 metadata

def load_cord19_metadata(path, max_docs=10_000, random_state=42):
    df = pd.read_csv(path, low_memory=False)

    # Different CORD-19 releases may use slightly different column names.
    title_col = "title" if "title" in df.columns else None
    abstract_col = "abstract" if "abstract" in df.columns else None

    if title_col is None:
        raise ValueError("Could not find a 'title' column in the CORD-19 metadata.")

    if abstract_col is None:
        df["abstract"] = ""
        abstract_col = "abstract"

    keep = [c for c in ["cord_uid", "sha", "title", "abstract", "publish_time", "journal"] if c in df.columns]
    df = df[keep].copy()

    df["title"] = df["title"].fillna("").astype(str)
    df["abstract"] = df["abstract"].fillna("").astype(str)

    # Use title + abstract as the retrieval text.
    df["text"] = (df["title"].str.strip() + ". " + df["abstract"].str.strip()).str.strip()

    # Remove records with almost no text.
    df = df[df["text"].str.len() >= 80].copy()

    # Remove exact duplicates.
    df = df.drop_duplicates(subset=["text"])

    if len(df) > max_docs:
        df = df.sample(max_docs, random_state=random_state)

    df = df.reset_index(drop=True)
    return df

if os.path.exists(CORD19_PATH):
    docs = load_cord19_metadata(CORD19_PATH, MAX_DOCS)
    print("Documents:", len(docs))
    display(docs[["title", "abstract"]].head(3))
else:
    docs = pd.DataFrame(columns=["title", "abstract", "text"])


Documents: 10000


,title,abstract
0,The utility of MEWS for predicting the mortali...,BACKGROUND: Older adults have been reported to...
1,Narrative trends of COVID-19 misinformation,The COVID-19 crisis has seen the rise of many ...
2,Generalized Multivariate Hawkes Processes,This work contributes to the theory and applic...



# Why dense retrieval?

Consider:

**Query**

> effects of coronavirus on lung tissue

**Relevant document**

> pulmonary damage caused by SARS-CoV-2 infection...

There may be little exact word overlap, although the meanings are closely related.

A sparse method such as TF-IDF/BM25 mainly exploits **term matching**.

Dense retrieval maps both the query and documents into a continuous vector space:

```text
query → encoder → q ∈ R^d

document → encoder → d ∈ R^d
```

Retrieval becomes a nearest-neighbour problem:

\[
d^* = \arg\max_d \; sim(q,d)
\]

Common similarity functions:

- cosine similarity
- dot product / inner product
- Euclidean distance

----

# Sparse vs dense vectors

Example vocabulary:

```text
[coronavirus, vaccine, cancer, treatment, patient]
```

A sparse representation might be:

```text
[0.7, 0.0, 0.0, 0.4, 0.2]
```

A dense embedding might look like:

```text
[0.21, -0.44, 0.17, 0.83, -0.12, ...]
```

A dense embedding has a fixed dimensionality, for example 384 dimensions for `all-MiniLM-L6-v2`.

The dimensions do **not** directly correspond to individual vocabulary words.



# Embeddings: from words to documents

## Word embeddings

Classical neural word embeddings include:

- Word2Vec
- GloVe
- FastText

These represent words as vectors.

However, word-level embeddings alone are not enough for document retrieval because:

- they do not directly represent an entire document;
- context can change meaning;
- long documents require aggregation.

## Contextual embeddings

Transformer models generate context-sensitive representations.

Examples:

- BERT
- RoBERTa
- DistilBERT

## Sentence/document embeddings

Models such as Sentence-BERT / Sentence Transformers are specifically trained to make semantically similar sentences close in vector space.

In [6]:
# Load a sentence embedding model

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

encoder = SentenceTransformer(MODEL_NAME)

print("Embedding dimension:", encoder.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_1131/2474845082.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", encoder.get_sentence_embedding_dimension())


In [7]:
# Small semantic similarity demonstration

sentences = [
    "A vaccine can protect people from infectious diseases.",
    "Vaccination provides protection against infections.",
    "The weather is expected to be sunny tomorrow.",
    "Deep learning models require training data."
]

sentence_embeddings = encoder.encode(
    sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

sim_matrix = sentence_embeddings @ sentence_embeddings.T

pd.DataFrame(
    sim_matrix,
    index=["S1", "S2", "S3", "S4"],
    columns=["S1", "S2", "S3", "S4"]
).round(3)


,S1,S2,S3,S4
S1,1.000,0.876,0.011,0.094
S2,0.876,1.000,0.015,0.086
S3,0.011,0.015,1.000,-0.055
S4,0.094,0.086,-0.055,1.000


## Similarity Functions

For two vectors \(q\) (query) and \(d\) (document), we can measure their similarity or distance using different functions.

### 1. Cosine Similarity

Cosine similarity measures the angle between two vectors:

$$
\text{cosine}(q,d) =
\frac{q \cdot d}{\|q\|_2 \|d\|_2}
$$


The value ranges from \(-1\) to \(1\), where a higher value indicates greater similarity.

---

### 2. Dot Product / Inner Product

The dot product between two vectors is:

$$
q \cdot d = \sum_{i=1}^{n} q_i d_i
$$

A larger dot product generally indicates greater similarity.

---

### 3. Euclidean Distance

Euclidean distance measures the straight-line distance between two vectors:

$$
\|q-d\|_2 =
\sqrt{\sum_{i=1}^{n}(q_i-d_i)^2}
$$

A **smaller distance** means the vectors are more similar.

---

### Relationship Between Cosine Similarity and Dot Product

If the embeddings are **L2-normalized**, then:

$$
\|q\|_2 = 1
$$

and

$$
\|d\|_2 = 1
$$

Therefore:

$$
\text{cosine}(q,d)
=
\frac{q \cdot d}{1 \times 1}
=
q \cdot d
$$

Hence:

$$
\boxed{\text{cosine similarity} = \text{dot product}}
$$

for normalized embeddings.

---

### Why is this important for FAISS?

FAISS provides an **Inner Product** index:

```python
faiss.IndexFlatIP(dimension)

In [8]:

# Compare similarity measures

from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

a = sentence_embeddings[0]
b = sentence_embeddings[1]

cos_sim = float(cosine_similarity([a], [b])[0][0])
dot_product = float(np.dot(a, b))
euclidean_distance = float(np.linalg.norm(a - b))

print("Cosine similarity :", round(cos_sim, 4))
print("Dot product       :", round(dot_product, 4))
print("Euclidean distance:", round(euclidean_distance, 4))


Cosine similarity : 0.8758
Dot product       : 0.8758
Euclidean distance: 0.4984



# 13. Dense retrieval: the bi-encoder architecture

The basic dense retriever has two encoders with the same embedding space:

```text
                 ┌──────────────────┐
Query ──────────►│ Query Encoder    │───► q
                 └──────────────────┘

                 ┌──────────────────┐
Document ───────►│ Document Encoder │───► d
                 └──────────────────┘

                         q ↔ d
                           │
                    similarity score
                           │
                         ranking
```

In [9]:
# Encode the CORD-19 collection

if len(docs) > 0:
    corpus_embeddings = encoder.encode(
        docs["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    print("Embedding matrix shape:", corpus_embeddings.shape)
else:
    corpus_embeddings = np.empty((0, encoder.get_sentence_embedding_dimension()), dtype="float32")
    print("Load CORD-19 first.")


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding matrix shape: (10000, 384)



# Method 1 — Brute-force dense retrieval

The simplest approach:

1. Encode all documents.
2. Encode the query.
3. Compute similarity with every document.
4. Sort by similarity.
5. Return top-k.

Complexity is approximately:

\[
O(Nd)
\]

for one query over \(N\) documents with embedding dimension \(d\).

This is excellent for teaching and small collections, but becomes expensive at very large scale.


In [10]:
# Brute-force dense retrieval

def dense_retrieve_bruteforce(query, documents, embeddings, encoder, k=5):
    q = encoder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores = embeddings @ q[0]
    top_indices = np.argsort(-scores)[:k]

    results = documents.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["rank"] = np.arange(1, len(results) + 1)

    return results[["rank", "score", "title", "abstract"]]

if len(docs) > 0:
    query = "effects of coronavirus infection on respiratory health"
    results = dense_retrieve_bruteforce(query, docs, corpus_embeddings, encoder, k=5)
    display(results)


,rank,score,title,abstract
4171,1,0.740124,A novel coronavirus (2019-nCoV) causing pneumo...,
7840,2,0.732816,Coronavirus genomic-sequence variations and th...,
1958,3,0.728888,Acute respiratory distress syndrome and steroi...,
6651,4,0.714356,Chapter 1 Respiratory virus and COVID-19,Viral infectious diseases remain a major chall...
2306,5,0.669460,A model of viral wheeze in nonasthmatic adults...,Episodic wheezing associated with viral infect...



# Method 2 — FAISS exact dense retrieval

FAISS = **Facebook AI Similarity Search**.

It provides efficient similarity search over dense vectors.

For normalized vectors, use:

```text
IndexFlatIP
```

where IP = Inner Product.

Because normalized inner product is equivalent to cosine similarity, this gives exact cosine-based nearest-neighbour search.


In [11]:
# Build a FAISS exact index

if len(corpus_embeddings) > 0:
    dimension = corpus_embeddings.shape[1]

    index_flat = faiss.IndexFlatIP(dimension)
    index_flat.add(corpus_embeddings)

    print("Number of indexed vectors:", index_flat.ntotal)
    print("Vector dimension:", index_flat.d)


Number of indexed vectors: 10000
Vector dimension: 384


In [12]:
# Search FAISS IndexFlatIP

def faiss_search(query, documents, encoder, index, k=5):
    q = encoder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q, k)

    result = documents.iloc[indices[0]].copy()
    result["score"] = scores[0]
    result["rank"] = np.arange(1, len(result) + 1)

    return result[["rank", "score", "title", "abstract"]]

if len(docs) > 0:
    results = faiss_search(
        "treatment options for COVID-19 patients",
        docs,
        encoder,
        index_flat,
        k=5
    )
    display(results)


,rank,score,title,abstract
3312,1,0.740015,Matched Cohort Study of Convalescent COVID-19 ...,
6625,2,0.739282,Comments for better understanding of the study...,
5194,3,0.727627,COVID-19 in bone marrow transplant recipients:...,
9498,4,0.727361,Pharmacological treatments of covid-19 - a review,Now the world stands in the fight against coro...
9785,5,0.717869,Antibiotics for the treatment of COVID-19,BACKGROUND: The effect of antibiotics with pot...



# Exact vs Approximate Nearest Neighbour (ANN)

For millions or billions of vectors, comparing a query with every vector can be expensive.

Dense retrieval therefore uses **Approximate Nearest Neighbour (ANN)** algorithms.

The key idea:

> Search a carefully selected part of the vector space rather than comparing against every vector.

Important ANN families:

- HNSW
- IVF
- IVF + PQ
- Product Quantization
- ScaNN
- Annoy
- graph-based search

---

# HNSW

HNSW = **Hierarchical Navigable Small World graph**.

Basic idea:

```text
vectors
   ↓
build graph
   ↓
connect nearby vectors
   ↓
start from an entry point
   ↓
navigate toward closer vectors
   ↓
return nearest neighbours
```

Advantages:

- very good retrieval quality
- fast ANN search
- no need to train a clustering model first

Trade-off:

- approximate search
- uses more memory than some compressed approaches


In [13]:
# FAISS HNSW demonstration

if len(corpus_embeddings) > 0:
    dimension = corpus_embeddings.shape[1]

    hnsw = faiss.IndexHNSWFlat(dimension, 32)
    hnsw.hnsw.efConstruction = 80
    hnsw.hnsw.efSearch = 64

    hnsw.add(corpus_embeddings)

    print("HNSW vectors:", hnsw.ntotal)

    hnsw_results = faiss_search(
        "long term effects of coronavirus infection",
        docs,
        encoder,
        hnsw,
        k=5
    )

    display(hnsw_results)


HNSW vectors: 10000


,rank,score,title,abstract
3457,1,0.726528,Potential long term adverse health outcomes of...,The coronavirus disease 2019(COVID-19) pandemi...
2149,2,0.751327,A NOVEL CORONAVIRUS MEETS THE CARDIOVASCULAR S...,
1814,3,0.802658,Clinical Characteristics of Patients With Prog...,Background: Coronavirus disease (COVID-19) has...
8428,4,0.804911,HIV and Human Coronavirus Coinfections: A Hist...,Seven human coronaviruses (hCoVs) are known to...
5974,5,0.812356,Clinical characteristics and short-term outcom...,INTRODUCTION: Since the first reported case of...



# 23. IVF — Inverted File Index for dense vectors

IVF partitions the vector space into clusters.

```text
All document vectors
        │
        ▼
   k-means clustering
        │
 ┌──────┼──────┐
 ▼      ▼      ▼
C1     C2     C3  ... Cn
```

At query time:

```text
Query
  ↓
find nearest clusters
  ↓
search only selected clusters
  ↓
return top-k
```

Important parameter:

`nprobe`

- small `nprobe` → faster, potentially lower recall
- large `nprobe` → slower, potentially higher recall

In [14]:
# FAISS IVF-Flat demonstration

if len(corpus_embeddings) >= 100:
    dimension = corpus_embeddings.shape[1]

    nlist = min(100, max(10, len(corpus_embeddings) // 100))

    quantizer = faiss.IndexFlatIP(dimension)
    ivf = faiss.IndexIVFFlat(
        quantizer,
        dimension,
        nlist,
        faiss.METRIC_INNER_PRODUCT
    )

    ivf.train(corpus_embeddings)
    ivf.add(corpus_embeddings)

    ivf.nprobe = min(10, nlist)

    print("IVF indexed vectors:", ivf.ntotal)
    print("nlist:", nlist)
    print("nprobe:", ivf.nprobe)

    ivf_results = faiss_search(
        "clinical symptoms associated with COVID-19",
        docs,
        encoder,
        ivf,
        k=5
    )

    display(ivf_results)
else:
    print("Need at least 100 documents for this demonstration.")


IVF indexed vectors: 10000
nlist: 100
nprobe: 10


,rank,score,title,abstract
4336,1,0.742138,Neurological symptoms of COVID-19 infection; a...,BACKGROUND: Coronavirus disease 2019 (COVID-19...
1629,2,0.735664,Haematological findings and pattern analysis i...,
5918,3,0.733515,Neurological and neuropsychiatric disorders as...,
4944,4,0.732109,Gastrointestinal symptoms associated with seve...,
3216,5,0.721809,"COVID-19 in Latin America: Symptoms, Morbiditi...",



# Dense retrieval and document chunking

A common RAG/retrieval pipeline is:

```text
Paper
  ↓
Text extraction
  ↓
Chunking
  ↓
Chunk 1 ──► embedding
Chunk 2 ──► embedding
Chunk 3 ──► embedding
...
  ↓
Vector index
```

Typical chunking strategies:

- fixed token/word length
- sentence-based chunking
- paragraph-based chunking
- overlapping chunks
- semantic chunking

### Why chunk?

A single vector for a 10-page paper can blur several unrelated topics.

Smaller chunks allow the retriever to find the exact passage relevant to the query.


In [15]:
# Simple sentence-based chunking demonstration

def simple_chunk(text, chunk_words=120, overlap=30):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks

if len(docs) > 0:
    example_text = docs.iloc[0]["text"]
    chunks = simple_chunk(example_text, chunk_words=80, overlap=20)

    print("Number of chunks:", len(chunks))
    for i, chunk in enumerate(chunks[:3], 1):
        print(f"\\n--- Chunk {i} ---\\n{chunk[:500]}...")


Number of chunks: 7
\n--- Chunk 1 ---\nThe utility of MEWS for predicting the mortality in the elderly adults with COVID-19: a retrospective cohort study with comparison to other predictive clinical scores. BACKGROUND: Older adults have been reported to be a population with high-risk of death in the COVID-19 outbreak. Rapid detection of high-risk patients is crucial to reduce mortality in this population. The aim of this study was to evaluate the prognositc accuracy of the Modified Early Warning Score (MEWS) for in-hospital mortality...
\n--- Chunk 2 ---\nstudy was to evaluate the prognositc accuracy of the Modified Early Warning Score (MEWS) for in-hospital mortality in older adults with COVID-19. METHODS: A retrospective cohort study was conducted in Wuhan Hankou Hospital in China from 1 January 2020 to 29 February 2020. Receiver operating characteristic (ROC) analysis was used to evaluate the predictive value of MEWS, Acute Physiology and Chronic Health Evaluation II (APACHE II), Se


# Query/document encoder choices

### Bi-encoder

```text
query ──► encoder ──► vector
document ──► encoder ──► vector

similarity(vector_q, vector_d)
```

Advantages:

- document embeddings can be precomputed
- scalable
- fast retrieval
- suitable for ANN indexing

### Cross-encoder

```text
(query, document)
       ↓
   Transformer
       ↓
   relevance score
```

The query and document are processed together.

Advantages:

- usually stronger relevance modelling

Disadvantage:

- expensive because every query-document pair must be processed

Therefore a common production architecture is:

```text
Millions of documents
        ↓
Dense bi-encoder retrieval
        ↓
Top 50 / Top 100
        ↓
Cross-encoder reranking
        ↓
Top 5 / Top 10
```


In [16]:
# Cross-encoder reranking demonstration


RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(RERANKER_NAME)

if len(docs) > 0:
    query = "treatment options for coronavirus patients"

    # First-stage dense retrieval
    candidate_k = 10
    candidates = faiss_search(
        query,
        docs,
        encoder,
        index_flat,
        k=candidate_k
    ).copy()

    pairs = [[query, text] for text in candidates["abstract"].fillna("").tolist()]

    rerank_scores = reranker.predict(pairs)

    candidates["rerank_score"] = rerank_scores
    candidates = candidates.sort_values(
        "rerank_score",
        ascending=False
    ).reset_index(drop=True)

    candidates["final_rank"] = np.arange(1, len(candidates) + 1)

    display(
        candidates[
            ["final_rank", "rerank_score", "score", "title"]
        ]
    )


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,final_rank,rerank_score,score,title
0,1,5.945178,0.718155,Experimental therapies under investigation for...
1,2,2.822920,0.689666,The current recommended drugs and strategies f...
2,3,2.242703,0.706547,Re-tasking the use of pre-existing medications...
3,4,-1.112645,0.672425,Antibiotics for the treatment of COVID-19
4,5,-1.149988,0.717893,COVID-19 Treatment: Close to a Cure? – A Rapid...
5,6,-2.078309,0.674332,Potential pharmacological and non-pharmacologi...
6,7,-8.394097,0.724686,A Study on Hospital Pharmacist’s Medical Suppo...
7,8,-8.394097,0.730526,Preventive and protective management for coron...
8,9,-8.394097,0.695238,"Multiple drugs: Coronavirus disease-2019, lack..."
9,10,-8.394097,0.685449,Recommendations for the management of newborn ...



# Dense retrieval evaluation

The same IR evaluation principles learned earlier still apply.

For a query:

```text
q → ranked documents
```

we can calculate:

- Precision@k
- Recall@k
- F1@k
- MRR
- MAP
- nDCG@k

Dense retrieval does **not** replace IR evaluation.

It changes how candidate documents are represented and retrieved.

---

## Relevance judgments

For a classroom experiment, create a small set of queries and relevant CORD-19 documents.

Example:

```python
qrels = {
    "vaccine efficacy": {"doc_1", "doc_7", "doc_13"},
    "coronavirus lung damage": {"doc_2", "doc_8"}
}
```

Then evaluate the ranking returned by the dense retriever.


In [ ]:

# 30. Generic Precision@k, Recall@k, MRR and AP utilities

def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return sum(doc in relevant for doc in retrieved_k) / k

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    if not relevant:
        return 0.0
    return sum(doc in relevant for doc in retrieved_k) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0

def average_precision(retrieved, relevant):
    if not relevant:
        return 0.0

    score = 0.0
    hits = 0

    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            hits += 1
            score += hits / rank

    return score / len(relevant)

print("Evaluation functions ready.")



# Hybrid retrieval — sparse + dense

Dense retrieval is powerful, but it is **not always superior** to lexical retrieval.

Sparse retrieval is often excellent for:

- exact names
- rare terminology
- identifiers
- numbers
- citations
- exact phrases

Dense retrieval is often excellent for:

- semantic similarity
- paraphrases
- synonyms
- conceptual matching

Therefore, many systems use **hybrid retrieval**.

```text
                     Query
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
       BM25 / sparse         Dense encoder
            │                     │
            ▼                     ▼
       Sparse ranking        Dense ranking
            │                     │
            └──────────┬──────────┘
                       ▼
                Score fusion
                       │
                       ▼
                 Final ranking
```

One simple fusion method:

$$
Score(d) = \alpha S_{dense}(d) + (1-\alpha)S_{sparse}(d)
$$


# Retrieval-Augmented Generation (RAG)

Dense retrieval answers the question:

> "Which documents/passages are relevant to my query?"

RAG adds a second component:

> "Given the retrieved information, can a language model generate an answer?"

The complete pipeline is:

```text
                    USER QUERY
                        │
                        ▼
                Query Embedding
                        │
                        ▼
                 Dense Retriever
                        │
                        ▼
                 FAISS Vector Index
                        │
                        ▼
                 Top-k Documents
                        │
                        ▼
              Retrieved Context
                        │
                        ▼
              Generative Language Model
                        │
                        ▼
                   FINAL ANSWER

In [18]:
# Install the generation libraries

!pip -q install -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 53.8 MB/s eta 0:00:00


In [19]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [20]:
# Load the generation model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-base"

generator_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    GEN_MODEL_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"

generator_model = generator_model.to(device)

print("Generation model loaded on:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generation model loaded on: cuda


In [21]:
question = "What are the main effects of COVID-19 on the respiratory system?"

prompt = f"""
Answer the following question concisely.

Question:
{question}

Answer:
"""

inputs = generator_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
).to(device)

outputs = generator_model.generate(
    **inputs,
    max_new_tokens=150
)

answer = generator_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", question)
print("\nGenerated answer:")
print(answer)

Question: What are the main effects of COVID-19 on the respiratory system?

Generated answer:
respiratory system


In [22]:
def retrieve_documents(query, k=5):
    query_embedding = encoder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index_flat.search(
        query_embedding,
        k
    )

    retrieved = docs.iloc[indices[0]].copy()

    retrieved["score"] = scores[0]
    retrieved["rank"] = range(1, len(retrieved) + 1)

    return retrieved.reset_index(drop=True)

In [23]:
query = "What are the effects of COVID-19 on the respiratory system?"

retrieved_docs = retrieve_documents(
    query,
    k=5
)

display(
    retrieved_docs[
        ["rank", "score", "title", "abstract"]
    ]
)

,rank,score,title,abstract
0,1,0.743600,Respiratory Pathophysiology of Mechanically Ve...,
1,2,0.672996,Late Breaking Abstract - 'Shielding': Patient ...,
2,3,0.668165,Proposal for Initiative of Evidence-based Trea...,The global spread of COVID-19 resulting in sev...
3,4,0.655408,,COVID-19 is rarely reported in children and th...
4,5,0.653503,NIH-Wide Strategic Plan for COVID-19 Research,From the Introduction: Coronavirus disease 201...


In [24]:
def build_context(retrieved_docs, max_chars=6000):

    context_parts = []
    current_length = 0

    for _, row in retrieved_docs.iterrows():

        title = str(row["title"])
        abstract = str(row["abstract"])

        passage = (
            f"Title: {title}\n"
            f"Abstract: {abstract}\n"
        )

        if current_length + len(passage) > max_chars:
            break

        context_parts.append(passage)
        current_length += len(passage)

    return "\n".join(context_parts)

In [25]:
context = build_context(
    retrieved_docs,
    max_chars=6000
)

print(context)

Title: Respiratory Pathophysiology of Mechanically Ventilated Patients with COVID-19: A Cohort Study
Abstract: 

Title: Late Breaking Abstract - 'Shielding': Patient experience of measures to reduce risk of COVID-19 in those with pre-existing respiratory conditions
Abstract: 

Title: Proposal for Initiative of Evidence-based Treatment of COVID-19 Patients with Worsening Hypoxia
Abstract: The global spread of COVID-19 resulting in severe Acute Respiratory Syndrome was classified by the WHO as pandemic in March 2020 Coronavirus (Covid-19) is an ongoing pandemic of coronavirus disease 2019 (COVID 19) caused by severe acute respiratory syndrome coronavirus 2 (SARS CoV 2) The disease was first identified in December 2019 in Wuhan, China The World Health Organization declared the outbreak a Public Health Emergency of International Concern on 30 January 2020 and a pandemic on 11 March As of 16 September 2020, more than 29 6 million cases have been reported in 188 countries and territories, re

In [26]:
def create_rag_prompt(question, context):

    prompt = f"""
You are a scientific question-answering assistant.

Answer the question using ONLY the information provided
in the context below.

If the context does not contain enough information to answer
the question, say that the retrieved documents do not provide
enough information.

Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [27]:
def generate_answer(prompt, max_new_tokens=200):

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    answer = generator_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [28]:
def rag_answer(question, k=5, max_context_chars=6000):

    # Step 1: Retrieve
    retrieved = retrieve_documents(
        question,
        k=k
    )

    # Step 2: Build context
    context = build_context(
        retrieved,
        max_chars=max_context_chars
    )

    # Step 3: Build prompt
    prompt = create_rag_prompt(
        question,
        context
    )

    # Step 4: Generate answer
    answer = generate_answer(
        prompt
    )

    return answer, retrieved

In [29]:
question = """
What are the major respiratory effects associated
with COVID-19 infection?
"""

answer, retrieved = rag_answer(
    question,
    k=5
)

print("=" * 80)
print("QUESTION")
print("=" * 80)

print(question)

print("\n" + "=" * 80)
print("RAG ANSWER")
print("=" * 80)

print(answer)

print("\n" + "=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, row in retrieved.iterrows():

    print(f"\n[{i+1}] {row['title']}")
    print(f"Similarity score: {row['score']:.4f}")

QUESTION

What are the major respiratory effects associated
with COVID-19 infection?


RAG ANSWER
severe and refractory hypoxia resulting from COVID-19 induced respiratory failure is the dominant cause of death

RETRIEVED SOURCES

[1] Respiratory Pathophysiology of Mechanically Ventilated Patients with COVID-19: A Cohort Study
Similarity score: 0.7449

[2] Proposal for Initiative of Evidence-based Treatment of COVID-19 Patients with Worsening Hypoxia
Similarity score: 0.6792

[3] Late Breaking Abstract - 'Shielding': Patient experience of measures to reduce risk of COVID-19 in those with pre-existing respiratory conditions
Similarity score: 0.6733

[4] Chapter 1 Respiratory virus and COVID-19
Similarity score: 0.6620

[5] Covid-19 in children: A brief overview after three months experience
Similarity score: 0.6579


In [30]:
question = "What are possible treatments for COVID-19?"

retrieved = retrieve_documents(
    question,
    k=5
)

print("TOP RETRIEVED DOCUMENTS")
print("=" * 80)

for _, row in retrieved.iterrows():

    print(f"\nRank: {row['rank']}")
    print(f"Score: {row['score']:.4f}")
    print(f"Title: {row['title']}")


context = build_context(
    retrieved,
    max_chars=6000
)

prompt = create_rag_prompt(
    question,
    context
)

answer = generate_answer(prompt)

print("\n\nGENERATED ANSWER")
print("=" * 80)
print(answer)

TOP RETRIEVED DOCUMENTS

Rank: 1
Score: 0.7129
Title: Are Medicinal Plants an Alternative Strategy to Combat COVID-19?

Rank: 2
Score: 0.6945
Title: Pharmacological treatments of covid-19 - a review

Rank: 3
Score: 0.6941
Title: Pharmacologic Correlates that Address the Clinicopathobiology of COVID-19

Rank: 4
Score: 0.6864
Title: Interventions for treatment of COVID-19: a protocol for a living systematic review with network meta-analysis including individual patient data (The LIVING Project)

Rank: 5
Score: 0.6735
Title: Therapeutic agents currently employed against Covid-19: an effort to control the pandemic


GENERATED ANSWER
pharmacological and other interventions for COVID-19.


In [36]:
question = """
What is the cure for COVID-19?
"""

answer, retrieved = rag_answer(
    question,
    k=5
)

print("Question:")
print(question)

print("\nRAG Answer:")
print(answer)

print("\nRetrieved documents:")

for _, row in retrieved.iterrows():
    print("\n-", row["title"])

Question:

What is the cure for COVID-19?


RAG Answer:
SARS-CoV-2 vaccine.

Retrieved documents:

- Are Medicinal Plants an Alternative Strategy to Combat COVID-19?

- Pharmacologic Correlates that Address the Clinicopathobiology of COVID-19

- Comments for better understanding of the study clinical characteristics and prognostic factors in COVID-19 patients aged ≥80 years

- Therapeutic Options for Treatment of COVID-19: A Review from Repur-posed Drugs to New Drug Targets

- Covid-19 reinfection is rare


In [37]:
question = "What is the cure for COVID-19?"

for k in [1, 3, 5, 10]:

    answer, retrieved = rag_answer(
        question,
        k=k
    )

    print("\n" + "=" * 80)
    print(f"k = {k}")
    print("=" * 80)

    print(answer)


k = 1
No

k = 3
SARS-CoV-2 vaccine.

k = 5
SARS-CoV-2 vaccine.

k = 10
A review of the various treatment options for COVID-19
